In [ ]:
import os
import sys
import math
import random
from pathlib import Path
from typing import Tuple

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import vit_b_16, ViT_B_16_Weights

from tqdm.auto import tqdm
from torch.utils.tensorboard import SummaryWriter

print(torch.__version__)

# Paths
data_root = Path('/Users/ductruong/Documents/GitHub/WoodTN/Go_that')
assert data_root.exists(), f"Dataset directory not found: {data_root}"
train_root = data_root / 'train'
val_root = data_root / 'val'
test_root = data_root / 'test'

# Hardware
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

# Reproducibility
SEED = 42
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
torch.manual_seed(SEED)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)

NUM_WORKERS = max(1, (os.cpu_count() or 2) - 1)
SAVE_DIR = Path('/Users/ductruong/Documents/GitHub/WoodTN/save_model')
SAVE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = SAVE_DIR / 'runs' / 'vit_b16_wood60'
LOG_DIR.mkdir(parents=True, exist_ok=True)

print('Device:', device)


In [ ]:
# Hyperparameters
IMG_SIZE = 384
BATCH_SIZE = 24
EPOCHS = 80
BASE_LR = 3e-4
WEIGHT_DECAY = 0.05
WARMUP_EPOCHS = 5
LABEL_SMOOTH = 0.1
MIXUP_ALPHA = 0.2
CUTMIX_ALPHA = 1.0
USE_MIXUP = True

# Early Stopping
ES_PATIENCE = 5
ES_MIN_DELTA = 1e-4

from collections import Counter
from torch.utils.data import WeightedRandomSampler
import numpy as np

# Transforms tuned for wood textures and varying zoom
train_tfms_opt = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
    transforms.RandAugment(num_ops=2, magnitude=10),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.25, saturation=0.2, hue=0.02),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.10), value='random'),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

val_tfms_opt = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.15)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

# Datasets: use separate train/val(/test) folders
assert train_root.exists() and val_root.exists(), "Expecting 'train' and 'val' subfolders under data_root."
train_ds_opt = datasets.ImageFolder(str(train_root), transform=train_tfms_opt)
val_ds_opt = datasets.ImageFolder(str(val_root), transform=val_tfms_opt)
num_classes = len(train_ds_opt.classes)

if test_root.exists():
    test_ds_opt = datasets.ImageFolder(str(test_root), transform=val_tfms_opt)
else:
    test_ds_opt = val_ds_opt  # evaluate on val if no test

# Class-balanced sampler for potential class imbalance
train_targets = [y for _, y in train_ds_opt.samples]
class_counts = Counter(train_targets)
class_weights = {c: 1.0 / max(1, class_counts[c]) for c in class_counts}
sample_weights = [class_weights[y] for y in train_targets]
sampler = WeightedRandomSampler(weights=torch.as_tensor(sample_weights, dtype=torch.double), num_samples=len(sample_weights), replacement=True)

train_loader_opt = DataLoader(train_ds_opt, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True)
val_loader_opt = DataLoader(val_ds_opt, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader_opt = DataLoader(test_ds_opt, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

len(train_ds_opt), len(val_ds_opt), len(test_ds_opt), class_counts

In [ ]:
# TensorBoard writer
writer = SummaryWriter(log_dir=str(LOG_DIR))
print('TensorBoard log dir:', LOG_DIR)

In [ ]:
# Model, optimizer, cosine schedule with warmup, EMA, label smoothing, mixup
weights = ViT_B_16_Weights.IMAGENET1K_V1
model_opt = vit_b_16(weights=weights)

in_features = model_opt.heads.head.in_features
model_opt.heads.head = nn.Linear(in_features, num_classes)
model_opt = model_opt.to(device)

# Label smoothing
criterion_opt = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
optimizer_opt = torch.optim.AdamW(model_opt.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)

# Cosine schedule with warmup
max_steps = math.ceil(len(train_loader_opt) * EPOCHS)
warmup_steps = max(1, math.ceil(len(train_loader_opt) * WARMUP_EPOCHS))

def lr_lambda(step):
    if step < warmup_steps:
        return float(step) / float(max(1, warmup_steps))
    progress = float(step - warmup_steps) / float(max(1, max_steps - warmup_steps))
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler_opt = torch.optim.lr_scheduler.LambdaLR(optimizer_opt, lr_lambda=lr_lambda)

# EMA weights
class EMA:
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.shadow = {}
        self.decay = decay
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()
    def update(self, model: nn.Module):
        for name, param in model.named_parameters():
            if param.requires_grad:
                new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                self.shadow[name] = new_average.clone()
    def apply_to(self, model: nn.Module):
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data = self.shadow[name].clone()
    def restore(self, model: nn.Module):
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.backup:
                param.data = self.backup[name].clone()
        self.backup = {}

ema = EMA(model_opt, decay=0.999)

# Mixup/CutMix utilities
rng = torch.Generator(device=device).manual_seed(SEED)

def rand_beta(alpha: float):
    if alpha <= 0:
        return 1.0
    return float(torch.distributions.Beta(alpha, alpha).sample((1,)).item())

def mixup_data(x, y, alpha):
    lam = rand_beta(alpha)
    index = torch.randperm(x.size(0), device=x.device, generator=rng)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def cutmix_data(x, y, alpha):
    lam = rand_beta(alpha)
    batch_size, _, H, W = x.size()
    index = torch.randperm(batch_size, device=x.device, generator=rng)
    cx = torch.randint(0, W, (1,), device=x.device, generator=rng).item()
    cy = torch.randint(0, H, (1,), device=x.device, generator=rng).item()
    w = int(W * math.sqrt(1 - lam))
    h = int(H * math.sqrt(1 - lam))
    x1 = max(cx - w // 2, 0)
    y1 = max(cy - h // 2, 0)
    x2 = min(cx + w // 2, W)
    y2 = min(cy + h // 2, H)
    x[:, :, y1:y2, x1:x2] = x[index, :, y1:y2, x1:x2]
    lam = 1 - ((x2 - x1) * (y2 - y1) / (W * H))
    y_a, y_b = y, y[index]
    return x, y_a, y_b, lam

use_amp_opt = device.type == 'cuda'
scaler_opt = torch.cuda.amp.GradScaler(enabled=use_amp_opt)


In [ ]:
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device, criterion: nn.Module | None = None) -> Tuple[float, float]:
    model.eval()
    correct = 0
    total = 0
    loss_sum = 0.0
    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            if criterion is not None:
                loss_sum += criterion(outputs, labels).item() * labels.size(0)
    acc = correct / max(1, total)
    avg_loss = loss_sum / max(1, total) if criterion is not None else float('nan')
    return acc, avg_loss

# Optimized training loop (mixup/cutmix + cosine + EMA + EarlyStopping)
best_val_opt = 0.0
best_path_opt = SAVE_DIR / 'vit_b16_wood60_opt_best.pth'

no_improve_epochs = 0
step = 0
for epoch in range(1, EPOCHS + 1):
    model_opt.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    pbar = tqdm(train_loader_opt, desc=f'[OPT] Epoch {epoch}/{EPOCHS}', leave=False)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer_opt.zero_grad(set_to_none=True)

        mixed = False
        if USE_MIXUP:
            if torch.rand(1).item() < 0.5 and MIXUP_ALPHA > 0:
                images, y_a, y_b, lam = mixup_data(images, labels, MIXUP_ALPHA)
                mixed = True
            elif CUTMIX_ALPHA > 0:
                images, y_a, y_b, lam = cutmix_data(images, labels, CUTMIX_ALPHA)
                mixed = True

        if use_amp_opt:
            with torch.cuda.amp.autocast():
                outputs = model_opt(images)
                loss = (lam * criterion_opt(outputs, y_a) + (1 - lam) * criterion_opt(outputs, y_b)) if mixed else criterion_opt(outputs, labels)
            scaler_opt.scale(loss).backward()
            scaler_opt.step(optimizer_opt)
            scaler_opt.update()
        else:
            outputs = model_opt(images)
            loss = (lam * criterion_opt(outputs, y_a) + (1 - lam) * criterion_opt(outputs, y_b)) if mixed else criterion_opt(outputs, labels)
            loss.backward()
            optimizer_opt.step()

        ema.update(model_opt)
        scheduler_opt.step()
        step += 1

        running_loss += loss.item() * labels.size(0)
        with torch.no_grad():
            preds = outputs.argmax(dim=1)
            if mixed:
                running_correct += (lam * (preds == y_a).sum().item() + (1 - lam) * (preds == y_b).sum().item())
            else:
                running_correct += (preds == labels).sum().item()
            running_total += labels.size(0)
        pbar.set_postfix({'loss': f"{running_loss / max(1, running_total):.4f}", 'acc': f"{running_correct / max(1, running_total):.4f}"})

    # Epoch metrics
    train_loss_epoch = running_loss / max(1, running_total)
    train_acc_epoch = running_correct / max(1, running_total)

    # Validation with EMA
    ema.apply_to(model_opt)
    val_acc, val_loss = evaluate(model_opt, val_loader_opt, device, criterion_opt)
    ema.restore(model_opt)

    # LR
    current_lr = optimizer_opt.param_groups[0]['lr']

    # TensorBoard logging
    if 'writer' in globals() and writer is not None:
        writer.add_scalar('train/loss', train_loss_epoch, epoch)
        writer.add_scalar('train/acc', train_acc_epoch, epoch)
        writer.add_scalar('val/loss', val_loss, epoch)
        writer.add_scalar('val/acc', val_acc, epoch)
        writer.add_scalar('meta/lr', current_lr, epoch)
        writer.add_scalar('meta/gen_gap', train_acc_epoch - val_acc, epoch)

    print(f"[OPT] Epoch {epoch}: train_loss={train_loss_epoch:.4f}, train_acc={train_acc_epoch:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

    if val_acc > best_val_opt + ES_MIN_DELTA:
        best_val_opt = val_acc
        no_improve_epochs = 0
        torch.save({'model_state': model_opt.state_dict(), 'val_acc': val_acc, 'epoch': epoch, 'classes': train_ds_opt.classes}, best_path_opt)
        print(f"Saved new OPT best to {best_path_opt}")
    else:
        no_improve_epochs += 1
        print(f"No improvement for {no_improve_epochs}/{ES_PATIENCE} epochs.")
        if no_improve_epochs >= ES_PATIENCE:
            print("Early stopping triggered.")
            break

print('Best OPT val acc:', best_val_opt)
if 'writer' in globals() and writer is not None:
    writer.flush()


In [ ]:
# Load best and evaluate on val/test set
ckpt = torch.load(SAVE_DIR / 'vit_b16_wood60_opt_best.pth', map_location=device)
model_opt.load_state_dict(ckpt['model_state'])
model_opt.eval()

val_acc, val_loss = evaluate(model_opt, val_loader_opt, device, criterion_opt)
print('Val acc (best):', f"{val_acc:.4f}")
print('Val loss (best):', f"{val_loss:.4f}")

test_acc, test_loss = evaluate(model_opt, test_loader_opt, device, criterion_opt)
print('Test acc:', f"{test_acc:.4f}")
print('Test loss:', f"{test_loss:.4f}")

# Save class names for reference
with open(SAVE_DIR / 'vit_b16_wood60_classes.txt', 'w') as f:
    for cls in train_ds_opt.classes:
        f.write(cls + '\n')

print('Done.')


In [ ]:
# CUDA/NVIDIA optimizations
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision('high')
    except Exception:
        pass
print('Device:', device)

In [ ]:
# TTA evaluation and confusion matrix
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

@torch.no_grad()
def predict_tta_flips(model: nn.Module, images: torch.Tensor) -> torch.Tensor:
    logits_sum = model(images)
    logits_sum += model(torch.flip(images, dims=[-1]))        # hflip
    logits_sum += model(torch.flip(images, dims=[-2]))        # vflip
    logits_sum += model(torch.flip(images, dims=[-2, -1]))    # hvflip
    return logits_sum / 4.0

@torch.no_grad()
def evaluate_tta(model: nn.Module, loader: DataLoader, device: torch.device) -> Tuple[list, list]:
    model.eval()
    all_preds, all_labels = [], []
    for images, labels in tqdm(loader, desc='TTA Eval', leave=False):
        images = images.to(device)
        labels = labels.to(device)
        logits = predict_tta_flips(model, images)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
    return all_preds, all_labels

# Evaluate EMA weights on test set with TTA
ema.apply_to(model_opt)
preds, labels = evaluate_tta(model_opt, test_loader_opt, device)
ema.restore(model_opt)

acc = (np.array(preds) == np.array(labels)).mean()
print('OPT Test Acc (TTA=flips):', f"{acc:.4f}")

cm = confusion_matrix(labels, preds)
print('Classification report:')
print(classification_report(labels, preds, target_names=train_ds_opt.classes))
